First, run adversarial_eval.py to generate the raw data for these plots. Then run this notebook to generate the plots.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request

urllib.request.urlretrieve(
    'https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf',
    'IBMPlexMono-Regular.ttf'
)
fe = font_manager.FontEntry(fname='IBMPlexMono-Regular.ttf', name='plexmono')
font_manager.fontManager.ttflist.append(fe)

plt.rcParams.update({
    'axes.facecolor': '#f5f4e9',
    'grid.color': '#AAAAAA',
    'axes.edgecolor': '#333333',
    'figure.facecolor': '#FFFFFF',
    'axes.grid': False,
    'axes.prop_cycle': plt.cycler('color', plt.cm.Dark2.colors),
    'font.family': fe.name,
    'figure.figsize': (3.5, 3.5 / 1.2),
    'ytick.left': True,
    'xtick.bottom': True,
    'figure.dpi': 300,
})
os.makedirs('paper_figs', exist_ok=True)

In [ ]:
SUMMARY_CSV   = '../adversarial/adversarial_summary.csv'
ALL_RESULTS   = '../../gcn_results/all_results'
RUN_DATE      = '2026-03-14'
CENSOR_REGION = 'above'

df = pd.read_csv(SUMMARY_CSV)
CENSOR_SPLITS = sorted(df['censor_split'].unique())
NOISE_TYPES   = ['ynoise', 'xnoise', 'omission']

print(f'{len(df)} rows | censor_splits: {CENSOR_SPLITS}')
df.head()

In [ ]:
LABELS = {
    'ynoise':   'Label noise',
    'xnoise':   'Feature noise',
    'omission': 'Omission',
}
NT_MAP = {'ynoise': 'ynoise', 'xnoise': 'xnoise', 'omission': 'omit'}


def xnoise_to_level(s):
    """Similarity range string '0.8-1.0' → noise level (1 - midpoint)."""
    a, b = map(float, str(s).split('-'))
    return round(1 - (a + b) / 2, 3)


def noise_param_to_level(noise_type, noise_param):
    if noise_type == 'xnoise':
        return xnoise_to_level(noise_param)
    return float(noise_param)


def highest_noise_param(df, noise_type):
    """Return the noise_param with the highest noise level for a given noise type."""
    params = df[df['noise_type'] == noise_type]['noise_param'].unique()
    return max(params, key=lambda p: noise_param_to_level(noise_type, p))


def clean_baseline(noise_type, split, metric='rmse'):
    """No-noise upper RMSE or corr from all_results dataframe JSON."""
    nt   = NT_MAP[noise_type]
    path = f'{ALL_RESULTS}/gcn_{nt}_results_split{split}_{CENSOR_REGION}/dataframe_{RUN_DATE}.json'
    ref  = pd.read_json(path)
    col  = 'upper rmse' if metric == 'rmse' else 'upper corr'
    return float(ref.iloc[0][col])  # row 0 = no-noise


def recovery_curve(df, noise_type, split, metric='upper_rmse'):
    """
    Uses highest noise level for noise_type.
    Returns (clean_frac, y, yerr) where:
      clean_frac=0 -> noised model performance
      clean_frac>0 -> finetuned model performance
    """
    param = highest_noise_param(df, noise_type)
    sub = df[
        (df['noise_type'] == noise_type) &
        (df['censor_split'] == split) &
        (df['noise_param'] == param)
    ]
    grp = sub.groupby('clean_frac').agg(
        nm=(f'noised_{metric}_mean',    'mean'),
        ns=(f'noised_{metric}_std',     'mean'),
        fm=(f'finetuned_{metric}_mean', 'mean'),
        fs=(f'finetuned_{metric}_std',  'mean'),
    ).reset_index()
    frac = grp['clean_frac'].values
    y    = np.where(np.isclose(frac, 0), grp['nm'].values, grp['fm'].values)
    yerr = np.where(np.isclose(frac, 0), grp['ns'].values, grp['fs'].values)
    return frac, y, yerr


def recovery_curve_by_param(df, noise_type, noise_param, metric='upper_rmse'):
    """Average across censor_splits for a given (noise_type, noise_param)."""
    sub = df[(df['noise_type'] == noise_type) & (df['noise_param'] == noise_param)]
    grp = sub.groupby('clean_frac').agg(
        nm=(f'noised_{metric}_mean',    'mean'),
        ns=(f'noised_{metric}_std',     'mean'),
        fm=(f'finetuned_{metric}_mean', 'mean'),
        fs=(f'finetuned_{metric}_std',  'mean'),
    ).reset_index()
    frac = grp['clean_frac'].values
    y    = np.where(np.isclose(frac, 0), grp['nm'].values, grp['fm'].values)
    yerr = np.where(np.isclose(frac, 0), grp['ns'].values, grp['fs'].values)
    return frac, y, yerr

## Plot 1 — Recovery curve (RMSE)
Sensitive region RMSE vs adversary clean fraction. One line per noise type, faceted by censor_split.  
Dotted reference line = clean (no-noise) baseline from original sweep.

In [ ]:
n = len(CENSOR_SPLITS)
fig, axs = plt.subplots(1, n, figsize=(4.5 * n, 4), sharey=True)
if n == 1:
    axs = [axs]

for i, split in enumerate(CENSOR_SPLITS):
    ax = axs[i]
    ax.set_title(f'{int(split * 100)}% sensitive data', fontsize=11)
    ax.set_xlabel('Adversary clean fraction', fontsize=9)
    if i == 0:
        ax.set_ylabel('Sensitive region RMSE', fontsize=10)

    for j, nt in enumerate(NOISE_TYPES):
        frac, y, yerr = recovery_curve(df, nt, split, 'upper_rmse')
        ax.plot(frac, y, marker='o', label=LABELS[nt], color=f'C{j}')
        ax.fill_between(frac, y - yerr, y + yerr, alpha=0.2, color=f'C{j}')
        try:
            ref = clean_baseline(nt, split, 'rmse')
            ax.axhline(ref, linestyle=':', color=f'C{j}', alpha=0.6, linewidth=1)
        except FileNotFoundError:
            pass
    ax.grid(True)

axs[0].legend(fontsize=8)
plt.tight_layout()
plt.savefig('paper_figs/adversarial_recovery_rmse.pdf', bbox_inches='tight')
plt.show()

## Plot 2 — Sensitive vs non-sensitive RMSE
Shows fine-tuning recovers the sensitive region specifically. One panel per noise type, averaged across censor_splits and noise_params.

In [ ]:
fig, axs = plt.subplots(1, len(NOISE_TYPES), figsize=(4.5 * len(NOISE_TYPES), 4), sharey=True)

for j, nt in enumerate(NOISE_TYPES):
    ax = axs[j]
    ax.set_title(LABELS[nt], fontsize=11)
    ax.set_xlabel('Adversary clean fraction', fontsize=9)
    if j == 0:
        ax.set_ylabel('RMSE', fontsize=10)

    sub = df[df['noise_type'] == nt]
    grp = sub.groupby('clean_frac').agg(
        n_um=('noised_upper_rmse_mean', 'mean'),
        n_us=('noised_upper_rmse_std',  'mean'),
        n_lm=('noised_lower_rmse_mean', 'mean'),
        n_ls=('noised_lower_rmse_std',  'mean'),
        f_um=('finetuned_upper_rmse_mean', 'mean'),
        f_us=('finetuned_upper_rmse_std',  'mean'),
        f_lm=('finetuned_lower_rmse_mean', 'mean'),
        f_ls=('finetuned_lower_rmse_std',  'mean'),
    ).reset_index()

    frac  = grp['clean_frac'].values
    mask0 = np.isclose(frac, 0)

    upper_y   = np.where(mask0, grp['n_um'].values, grp['f_um'].values)
    lower_y   = np.where(mask0, grp['n_lm'].values, grp['f_lm'].values)
    upper_err = np.where(mask0, grp['n_us'].values, grp['f_us'].values)
    lower_err = np.where(mask0, grp['n_ls'].values, grp['f_ls'].values)

    ax.plot(frac, upper_y, marker='x', color='C1', label='Sensitive')
    ax.fill_between(frac, upper_y - upper_err, upper_y + upper_err, alpha=0.2, color='C1')
    ax.plot(frac, lower_y, marker='^', color='C0', label='Non-sensitive')
    ax.fill_between(frac, lower_y - lower_err, lower_y + lower_err, alpha=0.2, color='C0')
    ax.grid(True)

axs[0].legend(fontsize=9)
plt.tight_layout()
plt.savefig('paper_figs/adversarial_sens_vs_nonsens_rmse.pdf', bbox_inches='tight')
plt.show()

## Plot 3 — Recovery curve (Spearman correlation)
Same structure as Plot 1 but using upper_corr.

In [ ]:
fig, axs = plt.subplots(1, n, figsize=(4.5 * n, 4), sharey=True)
if n == 1:
    axs = [axs]

for i, split in enumerate(CENSOR_SPLITS):
    ax = axs[i]
    ax.set_title(f'{int(split * 100)}% sensitive data', fontsize=11)
    ax.set_xlabel('Adversary clean fraction', fontsize=9)
    if i == 0:
        ax.set_ylabel('Sensitive region Spearman r', fontsize=10)

    for j, nt in enumerate(NOISE_TYPES):
        frac, y, yerr = recovery_curve(df, nt, split, 'upper_corr')
        ax.plot(frac, y, marker='o', label=LABELS[nt], color=f'C{j}')
        ax.fill_between(frac, y - yerr, y + yerr, alpha=0.2, color=f'C{j}')
        try:
            ref = clean_baseline(nt, split, 'corr')
            ax.axhline(ref, linestyle=':', color=f'C{j}', alpha=0.6, linewidth=1)
        except FileNotFoundError:
            pass
    ax.grid(True)

axs[0].legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('paper_figs/adversarial_recovery_corr.pdf', bbox_inches='tight')
plt.show()

## Plot 4 — Recovery by noise level
One panel per noise type. Lines colored by noise param level (heavier noise = harder recovery?).  
xnoise params converted to noise level via `1 - midpoint` (same convention as main_plots.ipynb).  
Averaged across censor_splits.

In [ ]:
fig, axs = plt.subplots(1, len(NOISE_TYPES), figsize=(4.5 * len(NOISE_TYPES), 4), sharey=False)

for j, nt in enumerate(NOISE_TYPES):
    ax = axs[j]
    ax.set_title(LABELS[nt], fontsize=11)
    ax.set_xlabel('Adversary clean fraction', fontsize=9)
    if j == 0:
        ax.set_ylabel('Sensitive region RMSE', fontsize=10)

    params = sorted(df[df['noise_type'] == nt]['noise_param'].unique())
    for k, param in enumerate(params):
        if nt == 'xnoise':
            level = xnoise_to_level(param)
            label = f'noise level {level}'
        else:
            label = f'noise = {param}'

        frac, y, yerr = recovery_curve_by_param(df, nt, param, 'upper_rmse')
        ax.plot(frac, y, marker='o', label=label, color=f'C{k}')
        ax.fill_between(frac, y - yerr, y + yerr, alpha=0.2, color=f'C{k}')

    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig('paper_figs/adversarial_by_noise_level.pdf', bbox_inches='tight')
plt.show()

## Plot 5 — Correlation vs noise level

X: noise level (scalar), Y: sensitive region Spearman r. One line per clean_frac, one panel per noise type. Averaged across censor_splits.

In [ ]:
fig, axs = plt.subplots(1, len(NOISE_TYPES), figsize=(4.5 * len(NOISE_TYPES), 4), sharey=True)

for j, nt in enumerate(NOISE_TYPES):
    ax = axs[j]
    ax.set_title(LABELS[nt], fontsize=11)
    ax.set_xlabel('Noise level', fontsize=9)
    if j == 0:
        ax.set_ylabel('Sensitive region Spearman r', fontsize=10)

    sub = df[df['noise_type'] == nt].copy()
    sub['noise_level'] = sub['noise_param'].apply(lambda p: noise_param_to_level(nt, p))

    for k, frac in enumerate(sorted(sub['clean_frac'].unique())):
        fsub = sub[sub['clean_frac'] == frac]
        grp = fsub.groupby('noise_level').agg(
            nm=('noised_upper_corr_mean',    'mean'),
            ns=('noised_upper_corr_std',     'mean'),
            fm=('finetuned_upper_corr_mean', 'mean'),
            fs=('finetuned_upper_corr_std',  'mean'),
        ).reset_index()

        levels = grp['noise_level'].values
        y    = np.where(np.isclose(frac, 0), grp['nm'].values, grp['fm'].values)
        yerr = np.where(np.isclose(frac, 0), grp['ns'].values, grp['fs'].values)

        ax.plot(levels, y, marker='o', label=f'clean_frac={frac}', color=f'C{k}')
        ax.fill_between(levels, y - yerr, y + yerr, alpha=0.15, color=f'C{k}')

    ax.grid(True)

axs[0].legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.savefig('paper_figs/adversarial_corr_vs_noise_level.pdf', bbox_inches='tight')
plt.show()

## Plot 6 — Correlation vs noise level (rows: noise type, columns: % sensitive data)
Same axes as Plot 5 but fully faceted — no averaging across censor_splits.

In [ ]:
noise_type_order = ['omission', 'xnoise', 'ynoise']
n_rows = len(noise_type_order)
n_cols = len(CENSOR_SPLITS)

fig, axs = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4 * n_rows), sharey='row')

for r, nt in enumerate(noise_type_order):
    sub = df[df['noise_type'] == nt].copy()
    sub['noise_level'] = sub['noise_param'].apply(lambda p: noise_param_to_level(nt, p))

    for c, split in enumerate(CENSOR_SPLITS):
        ax = axs[r, c]
        if r == 0:
            ax.set_title(f'{int(split * 100)}% sensitive data', fontsize=11)
        if c == 0:
            ax.set_ylabel(f'{LABELS[nt]}\nSpearman r', fontsize=9)
        if r == n_rows - 1:
            ax.set_xlabel('Noise level', fontsize=9)

        ssub = sub[sub['censor_split'] == split]
        for k, frac in enumerate(sorted(ssub['clean_frac'].unique())):
            fsub = ssub[ssub['clean_frac'] == frac]
            grp = fsub.groupby('noise_level').agg(
                nm=('noised_upper_corr_mean',    'mean'),
                ns=('noised_upper_corr_std',     'mean'),
                fm=('finetuned_upper_corr_mean', 'mean'),
                fs=('finetuned_upper_corr_std',  'mean'),
            ).reset_index()

            levels = grp['noise_level'].values
            y    = np.where(np.isclose(frac, 0), grp['nm'].values, grp['fm'].values)
            yerr = np.where(np.isclose(frac, 0), grp['ns'].values, grp['fs'].values)

            ax.plot(levels, y, marker='o', label=f'clean_frac={frac}', color=f'C{k}')
            ax.fill_between(levels, y - yerr, y + yerr, alpha=0.15, color=f'C{k}')

        ax.grid(True)

# single legend outside
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, fontsize=8, loc='center right', bbox_to_anchor=(1.08, 0.5))
plt.tight_layout()
plt.savefig('paper_figs/adversarial_corr_vs_noise_faceted.pdf', bbox_inches='tight')
plt.show()